# 고객 간 분석을 위한 Amazon Bedrock AgentCore Memory

## 개요

이 튜토리얼에서는 [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) namespace를 고객 간 분석에 사용하는 방법을 살펴봅니다. Part 1에서는 한 번에 한 고객을 대상으로 하는 개인화를 다뤘지만, 이 Notebook에서는 관점을 바꾸어 Admin/Marketing Agent가 **모든 고객**을 질의해 pattern을 찾고 funnel 효과를 측정하며 신제품을 추천합니다.

### 중요한 이유

기존 분석에는 ETL pipeline, data warehouse, 사전 구축된 dashboard가 필요합니다. AgentCore Memory namespace를 사용하면 Agent가 다음 작업을 수행할 수 있습니다.
- 사전 집계 table 없이 모든 고객을 동적으로 질의
- 지출, 선호도, 거절, 생애 event 등 여러 데이터 차원을 종합적으로 추론
- SQL 또는 BI 도구 전문 지식 없이 자연어로 insight 생성
- dashboard 구축 시 예상하지 못한 ad-hoc 질문에 답변


### 튜토리얼 세부 정보

| 정보         | 세부 정보                                                    |
|:--------------------|:-----------------------------------------------------------|
| 튜토리얼 유형       | Memory를 활용한 고객 간 분석                       |
| 기능             | 장기 메모리 Namespace + Strands Agent                |
| 주요 기능        | 고객 간 질의, Funnel 분석, 제품 격차 분석 |
| 예제 난이도  | 중급                                               |
| 사용 SDK            | boto3, bedrock-agentcore, strands-agents, strands-agents-tools |

### 학습 내용

1. 넓은 namespace prefix를 사용하여 모든 고객 질의
2. 알려진 고객을 순회하며 데이터를 집계하는 도구 구축
3. 추천 funnel 분석(pending → accepted → declined)
4. 거절 사유 및 충족되지 않은 선호도에서 제품 격차 식별
5. 고객 간 통계 분석에 `python_repl` 사용

### 아키텍처

![아키텍처](architecture.png)

### 작동 방식

Admin Agent는 Part 1과 동일한 namespace 구조를 사용하지만 질의 방식은 다릅니다.

```
Part 1 (customer-facing):  /bank/customers/customer_001/preferences     → one customer
Part 2 (admin analytics):  /bank/customers/{each_customer}/preferences  → aggregate all
```

Agent는 알려진 고객을 순회하며 각 고객의 데이터를 가져와 결합된 dataset을 종합적으로 추론합니다.

## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* AgentCore Memory 및 Amazon Bedrock 액세스 권한이 구성된 AWS 자격 증명
* Amazon Bedrock 모델 액세스(Claude Sonnet)

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip install "boto3>=1.42.63" "bedrock-agentcore[strands-agents]" strands-agents strands-agents-tools pandas

### 환경 설정

In [ ]:
import json
import os
import boto3
import time
import uuid
from datetime import datetime
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.client import MemoryClient
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from strands_tools import python_repl

# python_repl 확인 prompt 및 대화형 모드 억제
os.environ["BYPASS_TOOL_CONSENT"] = "true"
os.environ["PYTHON_REPL_INTERACTIVE"] = "false"

REGION = "us-west-2"
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"
KNOWN_CUSTOMERS = ["customer_001", "customer_002", "customer_003"]

agentcore_client = boto3.client("bedrock-agentcore", region_name=REGION)
memory_client = MemoryClient(region_name=REGION)
print(f"✅ Initialized clients for region: {REGION}")

## 1. Memory 인스턴스 생성 및 데이터 입력

이 Notebook은 독립적으로 실행되며 자체 Memory 인스턴스를 생성하고 3개의 고객 프로필을 나타내는 샘플 데이터를 입력합니다.

In [ ]:
unique_name = f"analytics_demo_{uuid.uuid4().hex[:8]}"
memory = memory_client.create_memory_and_wait(name=unique_name, strategies=[])
MEMORY_ID = memory["memoryId"]
print(f"✅ Created memory: {MEMORY_ID}")

### 샘플 데이터

Part 1에서 사용한 것과 동일한 다중 소스 고객 데이터를 입력합니다. 각 고객은 추천, 거래, 선호도, 생애 event의 네 가지 namespace 유형에 record를 가집니다.

In [ ]:
# 고객 001: 여행을 자주 하며 가격에 민감
customer_001_data = [
    {
        "namespace": "/bank/customers/customer_001/recommendations/pending",
        "content": {
            "product": "Travel Rewards Card",
            "annual_fee": 95,
            "benefits": "3x points on travel, 2x dining",
            "reason": "High travel spending detected",
        },
    },
    {
        "namespace": "/bank/customers/customer_001/recommendations/declined",
        "content": {
            "product": "Premium Platinum Card",
            "annual_fee": 495,
            "declined_reason": "Customer objected to annual fee",
            "declined_date": "2024-01-15",
        },
    },
    {
        "namespace": "/bank/customers/customer_001/transactions/summary",
        "content": {
            "monthly_avg": {
                "travel": 2400,
                "dining": 800,
                "groceries": 600,
                "gas": 200,
                "other": 500,
            },
            "total_monthly": 4500,
            "period": "last_6_months",
        },
    },
    {
        "namespace": "/bank/customers/customer_001/preferences",
        "content": {
            "max_annual_fee": 100,
            "priority_categories": ["travel", "dining"],
            "stated": "Prefers cards with low annual fees but good travel benefits",
        },
    },
    {
        "namespace": "/bank/customers/customer_001/life_events",
        "content": {
            "events": [
                {
                    "type": "upcoming_travel",
                    "details": "Japan trip booked for next month",
                    "detected": "2024-02-01",
                },
                {
                    "type": "spending_increase",
                    "category": "travel",
                    "change": "+40%",
                    "detected": "2024-01-15",
                },
            ]
        },
    },
]

# 고객 002: 고소득이며 premium 혜택을 중시
customer_002_data = [
    {
        "namespace": "/bank/customers/customer_002/recommendations/pending",
        "content": {
            "product": "Premium Platinum Card",
            "annual_fee": 495,
            "benefits": "5x all travel, lounge access, $300 travel credit",
            "reason": "High income segment, luxury spending patterns",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/recommendations/accepted",
        "content": {
            "product": "Business Rewards",
            "annual_fee": 150,
            "accepted_date": "2024-01-20",
            "reason": "Business expenses detected",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/transactions/summary",
        "content": {
            "monthly_avg": {
                "travel": 5000,
                "dining": 2000,
                "luxury": 3000,
                "business": 4000,
                "other": 1000,
            },
            "total_monthly": 15000,
            "period": "last_6_months",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/preferences",
        "content": {
            "max_annual_fee": "no_limit",
            "priority_categories": ["travel", "luxury", "business"],
            "stated": "Values premium benefits and status, fee is not a concern",
        },
    },
    {
        "namespace": "/bank/customers/customer_002/life_events",
        "content": {
            "events": [
                {
                    "type": "business_growth",
                    "details": "Business expenses up 60%",
                    "detected": "2024-02-01",
                }
            ]
        },
    },
]

# 고객 003: 학생이며 가격에 매우 민감
customer_003_data = [
    {
        "namespace": "/bank/customers/customer_003/recommendations/pending",
        "content": {
            "product": "No-Fee Starter Card",
            "annual_fee": 0,
            "benefits": "1% cashback on everything",
            "reason": "Alternative after fee objection",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/recommendations/declined",
        "content": {
            "product": "Student Card",
            "annual_fee": 95,
            "declined_reason": "Annual fee too high",
            "declined_date": "2024-02-01",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/transactions/summary",
        "content": {
            "monthly_avg": {
                "groceries": 300,
                "dining": 150,
                "entertainment": 100,
                "transport": 80,
                "other": 70,
            },
            "total_monthly": 700,
            "period": "last_6_months",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/preferences",
        "content": {
            "max_annual_fee": 0,
            "priority_categories": ["groceries", "dining"],
            "stated": "Student budget, absolutely no annual fees",
        },
    },
    {
        "namespace": "/bank/customers/customer_003/life_events",
        "content": {
            "events": [
                {
                    "type": "student",
                    "details": "University student",
                    "detected": "2023-09-01",
                }
            ]
        },
    },
]

all_data = customer_001_data + customer_002_data + customer_003_data
print(f"📊 Prepared {len(all_data)} records across 3 customers")

In [ ]:
records = []
current_time = datetime.now().timestamp()

for idx, item in enumerate(all_data):
    records.append(
        {
            "requestIdentifier": f"record_{idx:03d}",
            "namespaces": [item["namespace"]],
            "content": {"text": json.dumps(item["content"])},
            "timestamp": current_time + idx,
        }
    )

response = agentcore_client.batch_create_memory_records(memoryId=MEMORY_ID, records=records)

print(f"✅ Created {len(response['successfulRecords'])} records")
print("   Namespaces: recommendations, transactions, preferences, life_events")
print("   Customers: customer_001, customer_002, customer_003")

### Record indexing 대기

AgentCore Memory에서 record를 질의하려면 indexing 시간이 필요합니다. Record가 나타날 때까지 polling합니다.

In [ ]:
session_manager = MemorySessionManager(memory_id=MEMORY_ID, region_name=REGION)

print("Waiting for records to be indexed...")
for attempt in range(20):
    records = session_manager.list_long_term_memory_records(
        namespace_prefix="/bank/customers/customer_001/recommendations/pending",
        max_results=10,
    )
    if len(records) > 0:
        print(f"✅ Records indexed after {(attempt + 1) * 15}s")
        break
    print(f"  Attempt {attempt + 1}/20 - not ready yet, waiting 15s...")
    time.sleep(15)
else:
    print("⚠️ Records not yet indexed. Wait a bit longer and re-run this cell.")

# 모든 고객 검증
for cid in KNOWN_CUSTOMERS:
    r = session_manager.list_long_term_memory_records(namespace_prefix=f"/bank/customers/{cid}", max_results=100)
    print(f"  {cid}: {len(r)} records")

## 2. Admin Tool 정의

Part 1과의 핵심 차이는 Admin Tool이 한 고객의 namespace를 질의하는 대신 주어진 namespace 유형에 대해 **모든 고객**을 질의하고 결과를 집계한다는 점입니다.

In [ ]:
@tool
def query_all_customers(namespace_type: str) -> str:
    """Query a specific namespace type across ALL customers and aggregate results.

    Args:
        namespace_type: The type of data to query across all customers. Options:
            - 'recommendations/pending' - pending recommendations for all customers
            - 'recommendations/declined' - declined recommendations for all customers
            - 'recommendations/accepted' - accepted recommendations for all customers
            - 'transactions/summary' - spending summaries for all customers
            - 'preferences' - stated preferences for all customers
            - 'life_events' - detected life events for all customers
            - 'all' - all data for all customers

    Returns:
        JSON with aggregated data from all customers.
    """
    all_results = {}
    for customer_id in KNOWN_CUSTOMERS:
        if namespace_type == "all":
            prefix = f"/bank/customers/{customer_id}"
        else:
            prefix = f"/bank/customers/{customer_id}/{namespace_type}"

        records = session_manager.list_long_term_memory_records(namespace_prefix=prefix, max_results=100)
        all_results[customer_id] = [json.loads(r["content"]["text"]) for r in records]

    return json.dumps(
        {
            "namespace_queried": namespace_type,
            "customers_queried": len(KNOWN_CUSTOMERS),
            "data": all_results,
        },
        indent=2,
    )


print("✅ Defined query_all_customers tool")

## 3. Admin Analytics Agent 생성

System prompt는 Agent가 marketing analyst처럼 전체 고객에서 pattern, 격차, 기회를 찾도록 안내합니다.

In [ ]:
ADMIN_SYSTEM_PROMPT = """You are a marketing analytics agent for a bank's credit card division. You analyse customer data across the entire customer base to find patterns, measure effectiveness, and recommend business actions.

You have access to aggregated customer data via query_all_customers

When answering questions:
1. Query the relevant namespace(s)
2. Use python_repl for ALL calculations, aggregations, and statistical analysis
3. Identify patterns across customers — don't just list individual data points
4. Provide actionable business recommendations backed by data
5. When recommending new products, base it on gaps between what customers want and what was declined

You are analysing a portfolio of customers, not serving an individual.
"""

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)


def create_admin_agent():
    return Agent(
        model=model,
        tools=[query_all_customers, python_repl],
        system_prompt=ADMIN_SYSTEM_PROMPT,
        callback_handler=None,
    )


agent = create_admin_agent()
print("✅ Created admin analytics agent")

## 4. 고객 간 분석 질의

이러한 질의는 Agent가 여러 고객을 동시에 종합적으로 추론하는 과정을 보여 줍니다. 컨텍스트 혼입을 방지하기 위해 질의마다 새 Agent를 생성합니다.

In [ ]:
%%capture

# 질의 1: 추천 funnel 분석

response = agent(
    "Analyse our recommendation funnel. How many recommendations are pending, accepted, and declined across all customers? What's our conversion rate?"
)

In [ ]:
print(response)

In [ ]:
%%capture

# 질의 2: 고객이 거절하는 이유는 무엇인가?

response = agent(
    "What are the main reasons customers are declining our card recommendations? Are there common patterns?"
)

In [ ]:
print(response)

In [ ]:
%%capture
# 질의 3: 지출에 따른 고객 segmentation

response = agent(
    "Segment our customers by their spending patterns and preferences. Which segments are underserved by our current product lineup?"
)

In [ ]:
print(response)

In [ ]:
%%capture
# 질의 4: 제품 격차 분석 - 다음에 무엇을 구축해야 하는가?

response = agent(
    """Based on all customer data — their preferences, spending patterns, decline reasons, and life events — what credit card product should we launch next?

Give me a specific product proposal with target segment, key features, and pricing."""
)

In [ ]:
print(response)

## 요약

### 구현 내용

1. **고객 간 질의** - Admin Agent가 Part 1과 동일한 namespace 구조를 모든 고객에 대해 동시에 질의
2. **Funnel 분석** - 사전 구축된 dashboard 없이 추천 효과(pending → accepted → declined) 측정
3. **Pattern 감지** - LLM 추론으로 공통 거절 사유와 충분히 지원되지 않는 segment 탐색
4. **제품 격차 분석** - 선호도, 거절, 지출 데이터를 사용하여 다음에 구축할 제품 추천
5. **동적 코드 생성** - Agent가 필요하다고 판단한 통계 계산에 `python_repl` 사용

### 핵심 insight

Part 1에서 개별 고객을 지원한 것과 동일한 namespace 구조가 여기서는 포트폴리오 수준의 분석을 지원합니다. ETL, schema 변경, 새 data pipeline 없이 query pattern과 system prompt만 달라집니다.

동적 namespace 선택(Agent가 가져올 데이터 결정)과 동적 코드 생성(Agent가 분석 방법 결정)을 결합하면 기존 BI 아키텍처에 필요한 ETL pipeline, 사전 구축된 schema 또는 고정된 dashboard 없이 ad-hoc 분석을 수행할 수 있습니다.

## 5. 리소스 정리

이 튜토리얼에서 생성한 Memory 리소스를 정리합니다.

In [ ]:
try:
    memory_client.delete_memory_and_wait(memory_id=MEMORY_ID)
    print(f"✅ Deleted memory resource: {MEMORY_ID}")
except Exception as e:
    print(f"⚠️ Cleanup error: {e}")